In [33]:
from pathlib import Path
parent_directory = Path.cwd().parent.parent
print(parent_directory) 
type(parent_directory)

c:\Users\x286384\MerckGroup\Digitalización - Documents\DigitalProjectsLibrary\Planning


pathlib.WindowsPath

In [34]:
import pandas as pd
import numpy as np
from pathlib import Path
#reemplazar con la ruta de los archivos correcta
parent_directory = Path.cwd().parent.parent
FCST=pd.read_excel(parent_directory / "FCST Merck Abril 26.xlsx",sheet_name="Abril 2026")
PROD_DC_2=pd.read_excel(parent_directory / "PRODUCTOS DC.xlsx",sheet_name="Catalogo")

In [35]:
"""""
Module: P&G Forecast extraction 2
Purpose: extraer ## de jeringas para todos los meses por producto
Date: 30/07/2026
Author: J.Gonzalez
"""
logbalanceo=[]
#Columnas importantes
id_cols = ['SKUMERCK']

#Calculo de mes y año actual
Month_today=pd.to_datetime("today").month
Year_today=pd.to_datetime("today").year
demand_today=pd.to_datetime(str(Year_today) + "-01-" + str(Month_today), format="%Y-%m-%d")
demand_today=str(demand_today)[0:10]

#print(FCST.columns)
#FCST.columns = [pd.to_datetime(col, errors='coerce').strftime('%Y-%m') if pd.notna(pd.to_datetime(col, errors='coerce')) else col for col in FCST.columns]

matching_cols = []

for i in range(len(FCST.columns) - 1, -1, -1):  # from [-1] backwards
    col = FCST.columns[i]
    parsed = pd.to_datetime(col, errors='coerce')
    if pd.notna(parsed):
        matching_cols.append(col)
        if parsed.month <= Month_today and parsed.year <= Year_today:
            break  #
matching_cols = list(reversed(matching_cols))
#matching_cols = [pd.to_datetime(col, errors='coerce').strftime('%Y-%m') if pd.notna(pd.to_datetime(col, errors='coerce')) else col for col in matching_cols]
#Union de columnas id con columna demanda mes actual
#keep_cols = id_cols + matching_cols
FCST_month = FCST[id_cols + matching_cols].copy()

FCST_month = FCST_month.dropna(subset=['SKUMERCK']).reset_index(drop=True)

#Forecast de jeringas por mes actual en csv
FCST_month.to_csv("FCST_month.csv", index=False)

""""
#extraccion columnas de demanda del mes actual

#en caso de encontrar dos columnas con la misma fecha conserva la segunda que ya incluye el calculo total de piezas.
if len(matching_cols) >= 2:
    matching_cols = [matching_cols[-1]]

Union de columnas id con columna demanda mes actual
if matching_cols:
    keep_cols = id_cols + matching_cols
    FCST_month = FCST[keep_cols].copy()
else:
    FCST_month = FCST[id_cols].copy()
FCST_month = FCST_month.rename(columns={FCST_month.columns[-1]: 'Demanda'})
"""

'"\n#extraccion columnas de demanda del mes actual\n\n#en caso de encontrar dos columnas con la misma fecha conserva la segunda que ya incluye el calculo total de piezas.\nif len(matching_cols) >= 2:\n    matching_cols = [matching_cols[-1]]\n\nUnion de columnas id con columna demanda mes actual\nif matching_cols:\n    keep_cols = id_cols + matching_cols\n    FCST_month = FCST[keep_cols].copy()\nelse:\n    FCST_month = FCST[id_cols].copy()\nFCST_month = FCST_month.rename(columns={FCST_month.columns[-1]: \'Demanda\'})\n'

In [36]:
"""""
Module: Produccion por linea 2
Purpose: Separa los productos por familia, y despues por linea que utiliza dentro de cada mes esa familia especifica
Date: 30/07/2026
Author: J.Gonzalez
"""
PROD_DC_2 = PROD_DC_2.merge(FCST_month, on='SKUMERCK', how='left')
PROD_DC_2.drop(['SKUMERCK'], axis=1, inplace=True)
#PROD_DC_2_months = [col for col in PROD_DC_2.columns if col not in ['Nombre granel', 'Linea Granel 1', 'Linea Granel 2']]

#dicccionario para la suma del group
agg_dict = {col: 'sum' for col in PROD_DC_2.columns if col not in ['Nombre granel', 'Linea Granel 1', 'Linea Granel 2']}

#Group by
PROD_DC_2 = PROD_DC_2.groupby(['Nombre granel', 'Linea Granel 1', 'Linea Granel 2'], as_index=False).agg(agg_dict)

#Una columna para identificar DC
PROD_DC_2['Linea Granel 1'] = np.where(
    (PROD_DC_2['Linea Granel 1'] == 1) & (PROD_DC_2['Linea Granel 2'] == 1),
    'DC2',
    'DC1')
PROD_DC_2.drop(['Linea Granel 2'], axis=1, inplace=True)
PROD_DC_2 = PROD_DC_2.rename(columns={'Linea Granel 1': 'DC'})
PROD_DC_2.to_csv("PROD_DC_2.csv", index=False)
print(PROD_DC_2)


                              Nombre granel   DC  2026-07-28 00:00:00  \
0                      DCS NEUROBION 10,000  DC1             230400.0   
1                      DCS NEUROBION 10,000  DC2             230400.0   
2    DCS NEUROBION 10,000 DOUBLE FILTRATION  DC1                  0.0   
3                      DCS NEUROBION 25,000  DC1             125200.0   
4                    DEXABION DC BULK - MEX  DC1             115200.0   
5                    DEXABION DC BULK - MEX  DC2             230399.0   
6              DOLO NEUROBION DC BULK - MEX  DC1             115201.0   
7              DOLO NEUROBION DC BULK - MEX  DC2             728799.0   
8             DOLO NEUROBION FORTE DC - MEX  DC1             345600.0   
9             DOLO NEUROBION FORTE DC - MEX  DC2              77600.0   
10       DOLO NEUROBION FORTE DC BULK - MEX  DC1                  0.0   
11  SYR.NEUROBION 25000 DC BULK STEVA - GUA  DC1             220400.0   

    2026-08-29 00:00:00  2026-09-30 00:00:00  2026

In [38]:
"""""
Module: Balanceo mensual
Purpose: balancea DC2 mandando su residuo a DC1 para cada familia en cada mes 
Date: 30/07/2026
Author: J.Gonzalez
"""

jeringas=115200

familias= PROD_DC_2['Nombre granel'].unique()
familias_con_dc2=PROD_DC_2.loc[PROD_DC_2['DC'] == 'DC2', 'Nombre granel'].unique()
print(familias_con_dc2)

registro_residuos = []
for familia in familias_con_dc2:
    for col in agg_dict.keys():
        # Filter rows for this specific family
        mask_dc2 = (PROD_DC_2['Nombre granel'] == familia) & (PROD_DC_2['DC'] == 'DC2')
        mask_dc1 = (PROD_DC_2['Nombre granel'] == familia) & (PROD_DC_2['DC'] == 'DC1')

        # Get the DC2 value for this family and month
        dc2_val = PROD_DC_2.loc[mask_dc2, col].values[0]

        # Check if DC2 value is not divisible by jeringas
        if dc2_val % jeringas != 0:
            residue = dc2_val % jeringas
            registro_residuos.append((familia, col, residue))

            # Add the residue to DC1
            PROD_DC_2.loc[mask_dc1, col] = PROD_DC_2.loc[mask_dc1, col].values[0] + residue

            # Subtract the residue from DC2
            PROD_DC_2.loc[mask_dc2, col] = dc2_val - residue



DC1=PROD_DC_2[PROD_DC_2['DC'] == 'DC1'].copy()
DC2=PROD_DC_2[PROD_DC_2['DC'] == 'DC2'].copy()
PROD_DC_2.to_csv("PROD_DC_2_balanceado.csv", index=False)
DC1.to_csv("DC1_FCST.csv", index=False)
DC2.to_csv("DC2_FCST.csv", index=False)
registro_residuos.to_csv("registro_residuos.csv", index=False)


<StringArray>
[         'DCS NEUROBION 10,000',        'DEXABION DC BULK - MEX',
  'DOLO NEUROBION DC BULK - MEX', 'DOLO NEUROBION FORTE DC - MEX']
Length: 4, dtype: str


PermissionError: [Errno 13] Permission denied: 'PROD_DC_2_balanceado.csv'